# Synthetic Academic Risk Prediction Dataset

## Generator and Validation

This notebook generates and validates an empirically calibrated,
synthetic longitudinal student-performance dataset for academic
risk prediction.

The dataset is synthetic and is not intended to represent the
Kenyan student population statistically.

## 1. Setup

In [1]:
R.version.string

[1] "R version 4.4.0 (2024-04-24)"

In [2]:
library(tidyverse)
library(MASS)
library(psych)

set.seed(54321)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘MASS’


The following object is masked from ‘package:dplyr’:

    select



Attaching package: ‘psych’


The following objects are masked from ‘package:ggplot2’:

    %+%, alpha




## 2. Configuration

In [3]:
n_students <- 5000
n_subjects <- 9

subjects <- c(
  "Mathematics", "English", "Kiswahili", "Chemistry", "Biology",
  "Physics", "History", "Geography", "Religious_Education"
)

assessment_names <- c("T1M", "T1F", "T2M", "T2F", "T3M", "T3F")
prediction_points <- assessment_names[-6]

subject_mean <- c(
  Mathematics = 56.4, English = 58.7, Kiswahili = 61.0,
  Chemistry = 55.8, Biology = 55.5, Physics = 53.9,
  History = 53.0, Geography = 52.1, Religious_Education = 73.7
)

subject_sd <- c(
  Mathematics = 19.7, English = 20.1, Kiswahili = 17.9,
  Chemistry = 18.5, Biology = 18.5, Physics = 17.5,
  History = 15.6, Geography = 15.1, Religious_Education = 15.9
)

factor_loading <- c(
  Mathematics = 0.72, English = 0.69, Kiswahili = 0.87,
  Chemistry = 0.81, Biology = 0.81, Physics = 0.79,
  History = 0.68, Geography = 0.70, Religious_Education = 0.69
)

alpha <- 0.25
beta <- 0.85
delta <- 2.0
gamma <- 0.05

attendance_reference <- NULL

attendance_trajectory_prob <- c(Stable = 0.60, Improving = 0.20, Declining = 0.20)
attendance_slope_mean <- c(Stable = 0, Improving = 1.5, Declining = -3)
attendance_slope_sd <- 0.8
attendance_baseline_mean <- 90
attendance_baseline_sd <- 7
attendance_noise_sd <- 2

pass_mark <- 50
n_noise_features <- 3

stopifnot(
  length(subjects) == n_subjects,
  length(subject_mean) == n_subjects,
  length(subject_sd) == n_subjects,
  length(factor_loading) == n_subjects,
  identical(names(subject_mean), subjects),
  identical(names(subject_sd), subjects),
  identical(names(factor_loading), subjects),
  all(subject_sd > 0),
  all(factor_loading > 0 & factor_loading < 1),
  abs(sum(attendance_trajectory_prob) - 1) < 1e-8,
  all(attendance_trajectory_prob >= 0),
  abs(sum(attendance_slope_mean * 0 + attendance_trajectory_prob) - 1) < 1e-8,
  beta >= 0 && beta < 1,
  alpha > 0,
  is.finite(gamma) && gamma >= 0,
  is.finite(delta) && delta > 0,
  pass_mark >= 0 && pass_mark <= 100
)

tibble(
  subject = subjects,
  mean = subject_mean,
  sd = subject_sd,
  factor_loading = factor_loading
)

subject,mean,sd,factor_loading
<chr>,<dbl>,<dbl>,<dbl>
Mathematics,56.4,19.7,0.72
English,58.7,20.1,0.69
Kiswahili,61.0,17.9,0.87
Chemistry,55.8,18.5,0.81
Biology,55.5,18.5,0.81
Physics,53.9,17.5,0.79
History,53.0,15.6,0.68
Geography,52.1,15.1,0.70
Religious_Education,73.7,15.9,0.69


## 3. Student Population

In [4]:

# Generate student-level latent characteristics and noise variables
student_data <- tibble(
  student_id = sprintf("S%04d", seq_len(n_students)),
  academic_ability = rnorm(n_students),
  noise_1 = rnorm(n_students),
  noise_2 = rnorm(n_students),
  noise_3 = rnorm(n_students)
)

head(student_data)

student_id,academic_ability,noise_1,noise_2,noise_3
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
S0001,-0.1789007,-1.2417830,1.4369186,-1.9720314
S0002,-0.9280441,0.7750328,0.7632367,-0.5525773
S0003,-0.7840337,1.1608944,0.1351431,1.0442321
S0004,-1.6506005,-1.0224459,-0.1897825,-0.6366203
S0005,-0.4080665,-0.2842287,-2.0264125,-0.4035478
S0006,-1.0955294,-0.7750972,-1.1681021,0.6733093


## 4. Attendance Generation

In [5]:
# Generate longitudinal attendance and retain audit variables.
attendance_trajectory <- sample(
  names(attendance_trajectory_prob), n_students,
  replace = TRUE, prob = attendance_trajectory_prob
)

attendance_baseline <- rnorm(n_students, attendance_baseline_mean, attendance_baseline_sd)
attendance_slope <- rnorm(
  n_students, attendance_slope_mean[attendance_trajectory], attendance_slope_sd
)

attendance_data <- map_dfc(seq_along(assessment_names), function(t) {
  location <- attendance_baseline + attendance_slope * (t - 1)
  lower <- pnorm(0, location, attendance_noise_sd)
  upper <- pnorm(100, location, attendance_noise_sd)

  tibble(!!paste0("Attendance_", assessment_names[t]) :=
    qnorm(runif(n_students, lower, upper), location, attendance_noise_sd))
})

attendance_data <- bind_cols(
  tibble(
    student_id = student_data$student_id,
    attendance_trajectory = attendance_trajectory,
    attendance_baseline = attendance_baseline,
    attendance_slope = attendance_slope
  ),
  attendance_data
)

attendance_columns <- paste0("Attendance_", assessment_names)
attendance_matrix <- as.matrix(attendance_data[, attendance_columns])

# Calculate fixed population references for attendance effects and interaction scaling.
attendance_reference <- mean(attendance_matrix)
attendance_reference_sd <- sd(as.vector(attendance_matrix))

# Validate attendance bounds, finite values, and reference statistics.
stopifnot(
  all(is.finite(attendance_matrix)),
  all(attendance_matrix >= 0 & attendance_matrix <= 100),
  is.finite(attendance_reference),
  is.finite(attendance_reference_sd),
  attendance_reference_sd > 0
)

attendance_reference
attendance_reference_sd
head(attendance_data)

[1] 88.87254

[1] 7.946049

student_id,attendance_trajectory,attendance_baseline,attendance_slope,Attendance_T1M,Attendance_T1F,Attendance_T2M,Attendance_T2F,Attendance_T3M,Attendance_T3F
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
S0001,Stable,91.62527,0.1489983,88.69125,92.25757,91.38561,90.63421,93.54256,93.87444
S0002,Improving,85.19241,3.1848506,82.10367,84.56698,92.57606,95.75022,98.18288,99.86715
S0003,Stable,89.00347,1.5644999,89.71316,89.42142,92.27484,92.69986,96.43551,97.33179
S0004,Declining,95.67383,-3.2303601,94.21736,94.96319,89.20823,83.80328,85.90173,79.40578
S0005,Improving,85.40343,2.4335837,88.49813,84.45525,88.64080,93.02292,91.86603,98.15956
S0006,Declining,84.49847,-2.3881146,84.29364,81.04221,77.11821,75.65362,74.65740,68.78383


In [6]:
# Check attendance distributions and monitor boundary concentration across periods.
attendance_summary <- attendance_data %>%
  pivot_longer(
    all_of(attendance_columns),
    names_to = "assessment",
    values_to = "attendance"
  ) %>%
  group_by(assessment) %>%
  summarise(
    mean = mean(attendance),
    sd = sd(attendance),
    min = min(attendance),
    max = max(attendance),
    near_0_pct = mean(attendance <= 1),
    near_100_pct = mean(attendance >= 99),
    .groups = "drop"
  )

attendance_summary

assessment,mean,sd,min,max,near_0_pct,near_100_pct
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Attendance_T1F,89.42673,6.763708,64.03286,99.99979,0,0.0704
Attendance_T1M,89.72196,6.597867,61.58953,99.99906,0,0.0712
Attendance_T2F,88.71504,7.933104,56.98541,99.99925,0,0.0920
Attendance_T2M,89.14356,7.204422,60.51453,99.99900,0,0.0784
Attendance_T3F,87.87236,9.773449,49.26291,99.99974,0,0.1280
Attendance_T3M,88.35557,8.767473,54.41862,99.99963,0,0.1080


In [8]:
# Check whether attendance persists between successive assessments.
attendance_correlation <- tibble(
  transition = paste(assessment_names[-6], assessment_names[-1], sep = "_to_"),
  correlation = map2_dbl(
    attendance_columns[-6], attendance_columns[-1],
    ~ cor(attendance_data[[.x]], attendance_data[[.y]])
  )
)

attendance_correlation

# Check whether generated attendance changes follow the intended trajectory directions.
trajectory_validation <- attendance_data %>%
  group_by(attendance_trajectory) %>%
  summarise(
    n = n(),
    T1M_mean = mean(Attendance_T1M),
    T3F_mean = mean(Attendance_T3F),
    change = T3F_mean - T1M_mean,
    .groups = "drop"
  )

trajectory_validation

# Monitor near-boundary attendance separately for each trajectory.
attendance_boundary_by_trajectory <- attendance_data %>%
  pivot_longer(
    all_of(attendance_columns),
    names_to = "assessment",
    values_to = "attendance"
  ) %>%
  group_by(attendance_trajectory, assessment) %>%
  summarise(
    near_0_pct = mean(attendance <= 1),
    near_100_pct = mean(attendance >= 99),
    .groups = "drop"
  )

attendance_boundary_by_trajectory

transition,correlation
<chr>,<dbl>
T1M_to_T1F,0.8897041
T1F_to_T2M,0.8983201
T2M_to_T2F,0.9220822
T2F_to_T3M,0.9372089
T3M_to_T3F,0.9537579


attendance_trajectory,n,T1M_mean,T3F_mean,change
<chr>,<int>,<dbl>,<dbl>,<dbl>
Declining,960,89.85202,75.21989,-14.6321315
Improving,1055,89.70425,95.03261,5.3283651
Stable,2985,89.68639,89.41082,-0.2755677


attendance_trajectory,assessment,near_0_pct,near_100_pct
<chr>,<chr>,<dbl>,<dbl>
Declining,Attendance_T1F,0,0.039583333
Declining,Attendance_T1M,0,0.075000000
Declining,Attendance_T2F,0,0.006250000
Declining,Attendance_T2M,0,0.015625000
Declining,Attendance_T3F,0,0.002083333
Declining,Attendance_T3M,0,0.004166667
Improving,Attendance_T1F,0,0.108056872
Improving,Attendance_T1M,0,0.076777251
Improving,Attendance_T2F,0,0.195260664


## 5. Academic Score Generation

In [9]:
# Calculate residual SDs to preserve the target latent variance.
residual_sd <- subject_sd * sqrt(1 - factor_loading^2)

stopifnot(
  length(residual_sd) == n_subjects,
  all(is.finite(residual_sd)),
  all(residual_sd > 0),
  identical(names(residual_sd), subjects)
)

tibble(
  subject = subjects,
  target_sd = subject_sd,
  factor_loading = factor_loading,
  residual_sd = residual_sd
)

# Map latent scores to bounded 0–100 observed scores.
generate_observed_score <- function(latent_score, target_mean, target_sd) {
  lower_p <- pnorm(0, mean = target_mean, sd = target_sd)
  upper_p <- pnorm(100, mean = target_mean, sd = target_sd)

  latent_p <- pnorm(
    latent_score,
    mean = target_mean,
    sd = target_sd
  )

  qnorm(
    lower_p + latent_p * (upper_p - lower_p),
    mean = target_mean,
    sd = target_sd
  )
}

subject,target_sd,factor_loading,residual_sd
<chr>,<dbl>,<dbl>,<dbl>
Mathematics,19.7,0.72,13.671289
English,20.1,0.69,14.548568
Kiswahili,17.9,0.87,8.825626
Chemistry,18.5,0.81,10.848953
Biology,18.5,0.81,10.848953
Physics,17.5,0.79,10.729370
History,15.6,0.68,11.438109
Geography,15.1,0.70,10.783557
Religious_Education,15.9,0.69,11.508569


In [10]:
# Store latent and observed scores separately for all assessments.
latent_scores <- vector("list", length(assessment_names))
observed_scores <- vector("list", length(assessment_names))

names(latent_scores) <- assessment_names
names(observed_scores) <- assessment_names

# Reuse the fixed attendance reference SD calculated during attendance generation.
stopifnot(
  is.finite(attendance_reference_sd),
  attendance_reference_sd > 0
)

for (t in seq_along(assessment_names)) {
  assessment <- assessment_names[t]
  current_attendance <- attendance_data[[paste0("Attendance_", assessment)]]

  if (t > 1) {
    previous_averages <- vapply(
      observed_scores[seq_len(t - 1)],
      function(x) rowMeans(x[subjects]),
      numeric(n_students)
    )
    historical_failures <- rowSums(previous_averages < pass_mark)
  }

  latent_matrix <- observed_matrix <- matrix(
    NA_real_,
    nrow = n_students,
    ncol = n_subjects,
    dimnames = list(student_data$student_id, subjects)
  )

  for (s in seq_along(subjects)) {
    subject <- subjects[s]
    mu <- subject_mean[s]
    sigma <- subject_sd[s]
    lambda <- factor_loading[s]

    ability_effect <- lambda * sigma * student_data$academic_ability

    if (t == 1) {
      # T1M depends only on ability and residual variation.
      latent_score <- mu + ability_effect +
        rnorm(n_students, sd = residual_sd[s])

    } else {
      previous_latent <- latent_scores[[t - 1]][[subject]]

      attendance_effect <- alpha *
        (current_attendance - attendance_reference)

      previous_z <- (
        previous_latent - mu - ability_effect
      ) / sigma

      attendance_z <- (
        current_attendance - attendance_reference
      ) / attendance_reference_sd

      interaction_effect <- gamma * previous_z * attendance_z
      failure_effect <- -delta * historical_failures

      latent_score <- mu + ability_effect +
        beta * (previous_latent - mu - ability_effect) +
        attendance_effect + failure_effect +
        interaction_effect +
        rnorm(n_students, sd = residual_sd[s])
    }

    observed_score <- generate_observed_score(
      latent_score, mu, sigma
    )

    latent_matrix[, subject] <- latent_score
    observed_matrix[, subject] <- observed_score
  }

  latent_scores[[t]] <- as.data.frame(latent_matrix)
  observed_scores[[t]] <- as.data.frame(observed_matrix)
}

# Combine observed scores into one student-level dataset.
score_data <- tibble(student_id = student_data$student_id)

for (assessment in assessment_names) {
  scores <- observed_scores[[assessment]]
  names(scores) <- paste0(assessment, "_", names(scores))
  score_data <- bind_cols(score_data, as_tibble(scores))
}

head(score_data)

student_id,T1M_Mathematics,T1M_English,T1M_Kiswahili,T1M_Chemistry,T1M_Biology,T1M_Physics,T1M_History,T1M_Geography,T1M_Religious_Education,⋯,T3M_Religious_Education,T3F_Mathematics,T3F_English,T3F_Kiswahili,T3F_Chemistry,T3F_Biology,T3F_Physics,T3F_History,T3F_Geography,T3F_Religious_Education
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
S0001,72.30395,39.60103,61.82543,60.81134,47.41260,62.68844,44.89275,54.13617,75.77789,⋯,85.85205,42.01087091,6.97935565,49.0895016,65.2933125,41.7404331,83.70619739,84.146171,74.88505,82.143094
S0002,55.88898,51.82572,46.50589,33.18770,34.03791,50.44440,42.13500,36.09484,68.30096,⋯,54.08658,17.91276410,19.32027725,31.2931294,0.3580528,30.0406546,15.01846511,45.562476,48.15138,74.529732
S0003,40.38818,35.78580,39.40588,52.28298,34.66390,14.00810,35.83041,48.88391,61.39690,⋯,60.34801,9.02332017,19.25425866,16.3390825,0.3637296,16.0981659,21.46101196,3.607007,43.26934,69.980031
S0004,20.08945,14.84553,43.21408,42.49334,17.72775,41.49320,32.08599,31.62800,55.28975,⋯,32.67883,0.01478639,0.02601044,11.2906573,20.9811070,0.1395809,2.76120493,1.005798,11.55141,14.211642
S0005,51.28996,31.14939,62.35684,67.46242,25.42044,49.81339,49.76173,47.61538,46.11881,⋯,39.92328,5.07337471,34.59456839,49.2425909,25.3710385,30.6613079,20.13180011,23.335368,21.56468,8.034583
S0006,37.49589,37.20363,42.03790,19.57291,44.30551,33.91525,32.62598,59.21301,49.68636,⋯,15.55478,2.49349077,6.66971075,0.6676841,2.8676263,7.0240800,0.03011239,18.609284,38.86116,12.619247


In [12]:
# Verify that ability and residual components reconstruct target latent variance.
residual_variance_check <- tibble(
  subject = subjects,
  target_variance = subject_sd^2,
  reconstructed_variance =
    (factor_loading * subject_sd)^2 + residual_sd^2
) %>%
  mutate(difference = reconstructed_variance - target_variance)

residual_variance_check

stopifnot(max(abs(residual_variance_check$difference)) < 1e-10)

# Check T1M's relationship with ability and its observed score distributions.
t1m_ability_validation <- tibble(
  subject = subjects,
  ability_correlation = vapply(
    subjects,
    function(subject) cor(
      latent_scores[["T1M"]][[subject]],
      student_data$academic_ability
    ),
    numeric(1)
  )
)

t1m_ability_validation

subject,target_variance,reconstructed_variance,difference
<chr>,<dbl>,<dbl>,<dbl>
Mathematics,388.09,388.09,-5.684342e-14
English,404.01,404.01,0.000000e+00
Kiswahili,320.41,320.41,0.000000e+00
Chemistry,342.25,342.25,0.000000e+00
Biology,342.25,342.25,0.000000e+00
Physics,306.25,306.25,0.000000e+00
History,243.36,243.36,-2.842171e-14
Geography,228.01,228.01,-5.684342e-14
Religious_Education,252.81,252.81,2.842171e-14


subject,ability_correlation
<chr>,<dbl>
Mathematics,0.7224403
English,0.6888981
Kiswahili,0.8722151
Chemistry,0.8086788
Biology,0.8054880
Physics,0.7900489
History,0.6890731
Geography,0.7138761
Religious_Education,0.6882702


In [13]:
# Summarise score distributions and monitor boundary concentration.
score_distribution_validation <- map_dfr(assessment_names, function(assessment) {
  map_dfr(subjects, function(subject) {
    x <- observed_scores[[assessment]][[subject]]

    tibble(
      assessment = assessment,
      subject = subject,
      mean = mean(x),
      sd = sd(x),
      min = min(x),
      max = max(x),
      near_0 = mean(x <= 1),
      near_100 = mean(x >= 99)
    )
  })
})

score_distribution_validation

# Check baseline cross-subject correlations.
t1m_subject_correlations <- cor(
  as.matrix(observed_scores[["T1M"]][, subjects])
)

round(t1m_subject_correlations, 3)

assessment,subject,mean,sd,min,max,near_0,near_100
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
T1M,Mathematics,55.46250,18.86859,9.365286e-01,99.93470,0.0004,0.0022
T1M,English,57.34199,18.57471,9.010206e-01,99.93081,0.0002,0.0024
T1M,Kiswahili,60.15926,17.13603,1.076386e+00,99.88491,0.0000,0.0022
T1M,Chemistry,55.39479,17.88354,1.354203e+00,99.98920,0.0000,0.0006
T1M,Biology,55.06521,17.79330,1.404609e+00,99.84241,0.0000,0.0018
T1M,Physics,53.76797,17.02583,1.960911e+00,99.43169,0.0000,0.0006
T1M,History,53.11292,15.70766,1.219768e+00,99.98562,0.0000,0.0010
T1M,Geography,51.86989,14.99614,3.045939e+00,99.35449,0.0000,0.0002
T1M,Religious_Education,72.00842,14.29688,1.729015e+01,99.95140,0.0000,0.0074


,Mathematics,English,Kiswahili,Chemistry,Biology,Physics,History,Geography,Religious_Education
Mathematics,1.000,0.488,0.628,0.577,0.573,0.572,0.500,0.529,0.506
English,0.488,1.000,0.597,0.558,0.556,0.538,0.478,0.489,0.471
Kiswahili,0.628,0.597,1.000,0.706,0.703,0.682,0.593,0.620,0.601
Chemistry,0.577,0.558,0.706,1.000,0.655,0.643,0.561,0.575,0.558
Biology,0.573,0.556,0.703,0.655,1.000,0.632,0.563,0.570,0.547
Physics,0.572,0.538,0.682,0.643,0.632,1.000,0.548,0.567,0.545
History,0.500,0.478,0.593,0.561,0.563,0.548,1.000,0.505,0.480
Geography,0.529,0.489,0.620,0.575,0.570,0.567,0.505,1.000,0.501
Religious_Education,0.506,0.471,0.601,0.558,0.547,0.545,0.480,0.501,1.000


In [14]:
# Check that longitudinal residuals match the intended random-noise component.
longitudinal_mechanism_validation <- map_dfr(
  2:length(assessment_names),
  function(t) {
    assessment <- assessment_names[t]
    current_attendance <- attendance_data[[paste0("Attendance_", assessment)]]

    map_dfr(subjects, function(subject) {
      s <- match(subject, subjects)
      mu <- subject_mean[s]
      sigma <- subject_sd[s]
      lambda <- factor_loading[s]

      previous_latent <- latent_scores[[t - 1]][[subject]]
      current_latent <- latent_scores[[t]][[subject]]
      ability_effect <- lambda * sigma * student_data$academic_ability

      previous_assessments <- observed_scores[seq_len(t - 1)]
      previous_averages <- vapply(
        previous_assessments,
        function(x) rowMeans(x[subjects]),
        numeric(n_students)
      )
      historical_failures <- rowSums(previous_averages < pass_mark)

      previous_z <- (previous_latent - mu - ability_effect) / sigma
      attendance_z <- (
        current_attendance - attendance_reference
      ) / attendance_reference_sd

      systematic_component <- mu + ability_effect +
        beta * (previous_latent - mu - ability_effect) +
        alpha * (current_attendance - attendance_reference) -
        delta * historical_failures +
        gamma * previous_z * attendance_z

      residual <- current_latent - systematic_component

      tibble(
        assessment = assessment,
        subject = subject,
        residual_mean = mean(residual),
        residual_sd = sd(residual)
      )
    })
  }
)

longitudinal_mechanism_validation

assessment,subject,residual_mean,residual_sd
<chr>,<chr>,<dbl>,<dbl>
T1F,Mathematics,0.150743779,13.792239
T1F,English,-0.217563868,14.588145
T1F,Kiswahili,-0.255231328,8.630548
T1F,Chemistry,-0.096422352,10.897726
T1F,Biology,-0.081984984,10.758954
T1F,Physics,-0.068595189,10.851520
T1F,History,0.160007239,11.645543
T1F,Geography,0.038582844,10.582848
T1F,Religious_Education,0.028992191,11.528352


In [15]:
# Check attendance–performance associations after T1M.
attendance_performance_validation <- map_dfr(
  2:length(assessment_names),
  function(t) {
    assessment <- assessment_names[t]

    map_dfr(subjects, function(subject) {
      tibble(
        assessment = assessment,
        subject = subject,
        correlation = cor(
          attendance_data[[paste0("Attendance_", assessment)]],
          latent_scores[[t]][[subject]]
        )
      )
    })
  }
)

attendance_performance_validation

# Check attendance trajectories and boundary concentration.
trajectory_attendance_validation <- attendance_data %>%
  dplyr::select(attendance_trajectory, all_of(attendance_columns)) %>%
  pivot_longer(
    all_of(attendance_columns),
    names_to = "assessment",
    values_to = "attendance"
  ) %>%
  group_by(attendance_trajectory, assessment) %>%
  summarise(
    mean = mean(attendance),
    sd = sd(attendance),
    near_0 = mean(attendance <= 1),
    near_100 = mean(attendance >= 99),
    .groups = "drop"
  )

trajectory_attendance_validation

assessment,subject,correlation
<chr>,<chr>,<dbl>
T1F,Mathematics,0.08250613
T1F,English,0.08192954
T1F,Kiswahili,0.09322819
T1F,Chemistry,0.08545798
T1F,Biology,0.07045606
T1F,Physics,0.07376559
T1F,History,0.09231367
T1F,Geography,0.10040234
T1F,Religious_Education,0.08210092


attendance_trajectory,assessment,mean,sd,near_0,near_100
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
Declining,Attendance_T1F,87.08798,7.018806,0,0.039583333
Declining,Attendance_T1M,89.85202,6.653105,0,0.075000000
Declining,Attendance_T2F,81.14441,7.745692,0,0.006250000
Declining,Attendance_T2M,84.06363,7.377368,0,0.015625000
Declining,Attendance_T3F,75.21989,8.447100,0,0.002083333
Declining,Attendance_T3M,78.13901,8.023228,0,0.004166667
Improving,Attendance_T1F,91.05472,6.498814,0,0.108056872
Improving,Attendance_T1M,89.70425,6.620605,0,0.076777251
Improving,Attendance_T2F,93.25492,5.996153,0,0.195260664


In [16]:
# Check persistence between consecutive latent assessment scores.
longitudinal_persistence_validation <- map_dfr(
  2:length(assessment_names),
  function(t) {
    map_dfr(subjects, function(subject) {
      tibble(
        transition = paste(
          assessment_names[t - 1],
          assessment_names[t],
          sep = "_to_"
        ),
        subject = subject,
        correlation = cor(
          latent_scores[[t - 1]][[subject]],
          latent_scores[[t]][[subject]]
        )
      )
    })
  }
)

longitudinal_persistence_validation

transition,subject,correlation
<chr>,<chr>,<dbl>
T1M_to_T1F,Mathematics,0.8010749
T1M_to_T1F,English,0.7827950
T1M_to_T1F,Kiswahili,0.8936404
T1M_to_T1F,Chemistry,0.8527006
T1M_to_T1F,Biology,0.8502685
T1M_to_T1F,Physics,0.8386907
T1M_to_T1F,History,0.7874862
T1M_to_T1F,Geography,0.8017289
T1M_to_T1F,Religious_Education,0.7922937


## 6. Outcome and Risk

In [17]:
# Define academic risk using the mean T3F score across all nine subjects.
final_score_columns <- paste0("T3F_", subjects)

stopifnot(all(final_score_columns %in% names(score_data)))

score_data <- score_data %>%
  mutate(
    final_score = rowMeans(across(all_of(final_score_columns))),
    academic_risk = as.integer(final_score < pass_mark)
  )

In [18]:
# Validate the final outcome.
stopifnot(
  nrow(score_data) == n_students,
  n_distinct(score_data$student_id) == n_students,
  length(final_score_columns) == n_subjects,
  all(is.finite(score_data$final_score)),
  all(score_data$final_score >= 0 & score_data$final_score <= 100),
  all(score_data$academic_risk %in% c(0L, 1L))
)

final_outcome_validation <- score_data %>%
  summarise(
    n_students = n(),
    mean_final_score = mean(final_score),
    sd_final_score = sd(final_score),
    min_final_score = min(final_score),
    max_final_score = max(final_score),
    median_final_score = median(final_score),
    risk_count = sum(academic_risk),
    non_risk_count = sum(academic_risk == 0),
    risk_prevalence = mean(academic_risk),
    pass_mark = pass_mark
  )

final_outcome_validation

n_students,mean_final_score,sd_final_score,min_final_score,max_final_score,median_final_score,risk_count,non_risk_count,risk_prevalence,pass_mark
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>
5000,49.27659,22.41729,0.1812715,95.69144,55.14816,2137,2863,0.4274,50


In [19]:
# Verify the assessment-subject structure and prediction points.
expected_score_columns <- as.vector(
  outer(assessment_names, subjects, paste, sep = "_")
)

stopifnot(
  all(expected_score_columns %in% names(score_data)),
  length(expected_score_columns) == length(assessment_names) * n_subjects,
  identical(prediction_points, assessment_names[-length(assessment_names)]),
  !("T3F" %in% prediction_points)
)

## 8. Prediction Snapshots

In [20]:
# Create snapshots using only information available at each prediction point.
noise_columns <- paste0("noise_", seq_len(n_noise_features))

snapshots <- setNames(
  lapply(prediction_points, function(point) {
    point_index <- match(point, assessment_names)
    available_assessments <- assessment_names[seq_len(point_index)]

    score_columns <- unlist(
      lapply(available_assessments, function(a) {
        paste0(a, "_", subjects)
      }),
      use.names = FALSE
    )

    attendance_available <- paste0(
      "Attendance_", available_assessments
    )

    score_data %>%
      dplyr::select(student_id, all_of(score_columns)) %>%
      left_join(
        attendance_data %>%
          dplyr::select(student_id, all_of(attendance_available)),
        by = "student_id"
      ) %>%
      left_join(
        student_data %>%
          dplyr::select(student_id, all_of(noise_columns)),
        by = "student_id"
      ) %>%
      left_join(
        score_data %>% dplyr::select(student_id, academic_risk),
        by = "student_id"
      ) %>%
      mutate(prediction_point = point, .before = 1)
  }),
  prediction_points
)

# Inspect the number of rows and columns in each snapshot.
snapshot_summary <- purrr::imap_dfr(
  snapshots,
  function(data, point) {
    tibble(
      prediction_point = point,
      n_students = nrow(data),
      n_columns = ncol(data),
      risk_prevalence = mean(data$academic_risk)
    )
  }
)

snapshot_summary

prediction_point,n_students,n_columns,risk_prevalence
<chr>,<int>,<int>,<dbl>
T1M,5000,16,0.4274
T1F,5000,26,0.4274
T2M,5000,36,0.4274
T2F,5000,46,0.4274
T3M,5000,56,0.4274


In [21]:
# Validate snapshot sizes, unique students, required columns, and absence of future predictors.
for (point in prediction_points) {
  snapshot <- snapshots[[point]]
  point_index <- match(point, assessment_names)

  expected_scores <- unlist(
    lapply(assessment_names[seq_len(point_index)], function(a) {
      paste0(a, "_", subjects)
    }),
    use.names = FALSE
  )

  expected_attendance <- paste0(
    "Attendance_", assessment_names[seq_len(point_index)]
  )

  stopifnot(
    nrow(snapshot) == n_students,
    n_distinct(snapshot$student_id) == n_students,
    all(c(
      expected_scores,
      expected_attendance,
      noise_columns,
      "academic_risk"
    ) %in% names(snapshot)),
    !any(grepl("^T3F_", names(snapshot))),
    !any(grepl("^Attendance_T3F$", names(snapshot))),
    all(snapshot$academic_risk %in% c(0L, 1L)),
    !anyNA(snapshot$academic_risk)
  )
}

# Confirm that the target is identical across all snapshots.
stopifnot(
  all(vapply(
    snapshots,
    function(x) identical(x$academic_risk, score_data$academic_risk),
    logical(1)
  ))
)

# Review predictor columns in each snapshot.
snapshot_columns <- purrr::imap_dfr(
  snapshots,
  function(data, point) {
    tibble(
      prediction_point = point,
      predictor_count = ncol(data) - 2L,
      predictors = paste(
        setdiff(names(data), c("prediction_point", "student_id", "academic_risk")),
        collapse = ", "
      )
    )
  }
)

snapshot_columns

prediction_point,predictor_count,predictors
<chr>,<int>,<chr>
T1M,14,"T1M_Mathematics, T1M_English, T1M_Kiswahili, T1M_Chemistry, T1M_Biology, T1M_Physics, T1M_History, T1M_Geography, T1M_Religious_Education, Attendance_T1M, noise_1, noise_2, noise_3"
T1F,24,"T1M_Mathematics, T1M_English, T1M_Kiswahili, T1M_Chemistry, T1M_Biology, T1M_Physics, T1M_History, T1M_Geography, T1M_Religious_Education, T1F_Mathematics, T1F_English, T1F_Kiswahili, T1F_Chemistry, T1F_Biology, T1F_Physics, T1F_History, T1F_Geography, T1F_Religious_Education, Attendance_T1M, Attendance_T1F, noise_1, noise_2, noise_3"
T2M,34,"T1M_Mathematics, T1M_English, T1M_Kiswahili, T1M_Chemistry, T1M_Biology, T1M_Physics, T1M_History, T1M_Geography, T1M_Religious_Education, T1F_Mathematics, T1F_English, T1F_Kiswahili, T1F_Chemistry, T1F_Biology, T1F_Physics, T1F_History, T1F_Geography, T1F_Religious_Education, T2M_Mathematics, T2M_English, T2M_Kiswahili, T2M_Chemistry, T2M_Biology, T2M_Physics, T2M_History, T2M_Geography, T2M_Religious_Education, Attendance_T1M, Attendance_T1F, Attendance_T2M, noise_1, noise_2, noise_3"
T2F,44,"T1M_Mathematics, T1M_English, T1M_Kiswahili, T1M_Chemistry, T1M_Biology, T1M_Physics, T1M_History, T1M_Geography, T1M_Religious_Education, T1F_Mathematics, T1F_English, T1F_Kiswahili, T1F_Chemistry, T1F_Biology, T1F_Physics, T1F_History, T1F_Geography, T1F_Religious_Education, T2M_Mathematics, T2M_English, T2M_Kiswahili, T2M_Chemistry, T2M_Biology, T2M_Physics, T2M_History, T2M_Geography, T2M_Religious_Education, T2F_Mathematics, T2F_English, T2F_Kiswahili, T2F_Chemistry, T2F_Biology, T2F_Physics, T2F_History, T2F_Geography, T2F_Religious_Education, Attendance_T1M, Attendance_T1F, Attendance_T2M, Attendance_T2F, noise_1, noise_2, noise_3"
T3M,54,"T1M_Mathematics, T1M_English, T1M_Kiswahili, T1M_Chemistry, T1M_Biology, T1M_Physics, T1M_History, T1M_Geography, T1M_Religious_Education, T1F_Mathematics, T1F_English, T1F_Kiswahili, T1F_Chemistry, T1F_Biology, T1F_Physics, T1F_History, T1F_Geography, T1F_Religious_Education, T2M_Mathematics, T2M_English, T2M_Kiswahili, T2M_Chemistry, T2M_Biology, T2M_Physics, T2M_History, T2M_Geography, T2M_Religious_Education, T2F_Mathematics, T2F_English, T2F_Kiswahili, T2F_Chemistry, T2F_Biology, T2F_Physics, T2F_History, T2F_Geography, T2F_Religious_Education, T3M_Mathematics, T3M_English, T3M_Kiswahili, T3M_Chemistry, T3M_Biology, T3M_Physics, T3M_History, T3M_Geography, T3M_Religious_Education, Attendance_T1M, Attendance_T1F, Attendance_T2M, Attendance_T2F, Attendance_T3M, noise_1, noise_2, noise_3"


In [22]:
# Save each prediction-point snapshot as a separate CSV file.
walk2(
  snapshots,
  names(snapshots),
  ~ write_csv(.x, paste0("snapshot_", .y, ".csv"))
)

## 9. Sensitivity Analysis

In [23]:
# Store sensitivity results for comparison across parameter settings.
sensitivity_results <- tibble()

capture_sensitivity_results <- function(round_name, changed_parameter, value) {
  adjacent_correlations <- map_dbl(
    2:length(assessment_names),
    function(t) {
      mean(vapply(
        subjects,
        function(subject) cor(
          latent_scores[[t - 1]][[subject]],
          latent_scores[[t]][[subject]]
        ),
        numeric(1)
      ))
    }
  )

  attendance_correlations <- map_dbl(
    2:length(assessment_names),
    function(t) {
      mean(vapply(
        subjects,
        function(subject) cor(
          attendance_data[[paste0("Attendance_", assessment_names[t])]],
          latent_scores[[t]][[subject]]
        ),
        numeric(1)
      ))
    }
  )

  tibble(
    round = round_name,
    changed_parameter = changed_parameter,
    tested_value = value,
    alpha = alpha,
    beta = beta,
    delta = delta,
    gamma = gamma,
    mean_final_score = mean(score_data$final_score),
    sd_final_score = sd(score_data$final_score),
    risk_prevalence = mean(score_data$academic_risk),
    mean_latent_persistence = mean(adjacent_correlations),
    mean_attendance_score_correlation = mean(attendance_correlations),
    attendance_near_100 = mean(
      as.matrix(attendance_data[, attendance_columns]) >= 99
    ),
    score_near_0 = mean(
      as.matrix(score_data[, unlist(lapply(
        assessment_names,
        function(a) paste0(a, "_", subjects)
      ))]) <= 1
    ),
    score_near_100 = mean(
      as.matrix(score_data[, unlist(lapply(
        assessment_names,
        function(a) paste0(a, "_", subjects)
      ))]) >= 99
    )
  )
}

In [24]:
sensitivity_results <- bind_rows(
  sensitivity_results,
  capture_sensitivity_results("Round 4", "gamma", gamma)
)

sensitivity_results

round,changed_parameter,tested_value,alpha,beta,delta,gamma,mean_final_score,sd_final_score,risk_prevalence,mean_latent_persistence,mean_attendance_score_correlation,attendance_near_100,score_near_0,score_near_100
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Round 4,gamma,0.05,0.25,0.85,2,0.05,49.27659,22.41729,0.4274,0.8790819,0.1758986,0.09133333,0.01436296,0.01097778


In [25]:
# Compare each experiment with the baseline.
baseline_metrics <- sensitivity_results %>%
  filter(round == "Baseline") %>%
  slice(1)

sensitivity_comparison <- sensitivity_results %>%
  mutate(
    change_mean_final_score =
      mean_final_score - baseline_metrics$mean_final_score,
    change_risk_prevalence =
      risk_prevalence - baseline_metrics$risk_prevalence,
    change_latent_persistence =
      mean_latent_persistence - baseline_metrics$mean_latent_persistence,
    change_attendance_correlation =
      mean_attendance_score_correlation -
      baseline_metrics$mean_attendance_score_correlation,
    change_attendance_boundary =
      attendance_near_100 - baseline_metrics$attendance_near_100
  ) %>%
  dplyr::select(
    round, changed_parameter, tested_value,
    alpha, beta, delta, gamma,
    change_mean_final_score, change_risk_prevalence,
    change_latent_persistence, change_attendance_correlation,
    change_attendance_boundary
  )

sensitivity_comparison

ERROR: [1m[33mError[39m in `mutate()`:[22m
[1m[22m[36mℹ[39m In argument: `change_mean_final_score = mean_final_score -
  baseline_metrics$mean_final_score`.
[1mCaused by error:[22m
[1m[22m[33m![39m `change_mean_final_score` must be size 1, not 0.
